In [10]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, from_json, unbase64, from_unixtime, udf
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, IntegerType, LongType, FloatType

# Create the Spark session
spark = SparkSession\
    .builder\
    .appName("Classified Streamer")\
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .config("spark.sql.debug.maxToStringFields", "1000")  \
    .getOrCreate()

In [11]:
payload_schema = StructType([
    StructField("before", StructType([
        StructField("order_id", IntegerType(), True),
        StructField("order_date", IntegerType(), True),
        StructField("product_name", StringType(), True),
        StructField("quantity_sold", IntegerType(), True),
        StructField("total_sales", StringType(), True),
        StructField("promotion", StringType(), True),
        StructField("region", StringType(), True),
        StructField("unit_price", StringType(), True)
    ])),
    StructField("after", StructType([
        StructField("order_id", IntegerType(), True),
        StructField("order_date", IntegerType(), True),
        StructField("product_name", StringType(), True),
        StructField("quantity_sold", IntegerType(), True),
        StructField("total_sales", StringType(), True),
        StructField("promotion", StringType(), True),
        StructField("region", StringType(), True),
        StructField("unit_price", StringType(), True)
    ])),
    StructField("source", StructType([
        StructField("version", StringType(), True),
        StructField("connector", StringType(), True),
        StructField("name", StringType(), True),
        StructField("ts_ms", LongType(), True),
        StructField("snapshot", StringType(), True),
        StructField("db", StringType(), True),
        StructField("sequence", StringType(), True),
        StructField("schema", StringType(), True),
        StructField("table", StringType(), True),
        StructField("txId", LongType(), True),
        StructField("lsn", LongType(), True),
        StructField("xmin", LongType(), True)
    ])),
    StructField("op", StringType(), True),
    StructField("ts_ms", LongType(), True),
    StructField("transaction", StringType(), True)
])

schema = StructType([
    StructField("payload", payload_schema)
])

In [12]:
# Define a UDF to decode binary (from unbase64) to a float, assuming scale = 2.
def decode_decimal(binary_val):
    if binary_val is None:
        return None
    try:
        # Convert the binary value to an integer (big-endian, signed)
        int_val = int.from_bytes(binary_val, byteorder="big", signed=True)
        # Divide by 100 (scale=2)
        return float(int_val) / 100.0
    except Exception as e:
        return None

decode_decimal_udf = udf(decode_decimal, FloatType())

In [13]:
# Read the streaming data from Redpanda (Kafka)
df = spark \
    .readStream \
    .format("kafka") \
    .option("kafka.bootstrap.servers", "redpanda:9092") \
    .option("subscribe", "dbz.public.orders") \
    .option("startingOffsets", "earliest") \
    .load()

In [14]:
# Decode the Kafka message from binary to string and parse it as JSON using the schema
df_parsed = df.selectExpr("CAST(value AS STRING) as json_str") \
    .select(from_json(col("json_str"), schema).alias("data"))

In [15]:
# Flatten the payload: select fields from "after" and some from "source"
df_flat = df_parsed.select(
    col("data.payload.after.order_id").alias("order_id"),
    col("data.payload.after.order_date").alias("order_date"),
    col("data.payload.after.product_name").alias("product_name"),
    col("data.payload.after.quantity_sold").alias("quantity_sold"),
    
    # Decode the base64-encoded fields and convert to float using the UDF
    decode_decimal_udf(unbase64(col("data.payload.after.total_sales"))).alias("total_sales"),
    decode_decimal_udf(unbase64(col("data.payload.after.unit_price"))).alias("unit_price"),
    
    
    col("data.payload.after.promotion").alias("promotion"),
    col("data.payload.after.region").alias("region"),
    
    col("data.payload.source.table").alias("source_table"),
    col("data.payload.source.db").alias("source_db"),
    col("data.payload.op").alias("operation"),
    col("data.payload.ts_ms").alias("event_timestamp"),
)
# Convert order_date from days since epoch to a timestamp (assuming order_date is in days)
df_flat = (
    df_flat
    .withColumn("order_date", from_unixtime(col("order_date") * 86400).cast("timestamp"))
    .withColumn("event_timestamp",from_unixtime(col("event_timestamp") / 1000).cast("timestamp"))
)

In [16]:
df_flat.writeStream \
    .format("delta") \
    .trigger(once=True) \
    .outputMode("append") \
    .option("checkpointLocation", "checkpoint") \
    .start("orders/data") \
    .awaitTermination()